# Pilot 9 批量迁移实验：评估器问题诊断与修复前基线

- 记录日期：2026-09-24
- 实验批次：`pilot_9`
- 模型：`qwen-plus`（provider: `qwen`）
- 最大修复轮数：3
- 本文档性质：修复评估器前的不可变诊断记录，不作为最终效果结论。

本轮原计划依次运行 3 个 simple、3 个 medium、3 个 complex。simple 与 medium 共 6 个样本完成后，发现评估器会把人工确认过的参考 San 也判为失败，因此暂停 complex，避免继续消耗 token 并产生不可比较数据。

## 1. 可追溯性与冻结快照

修复前全部实验产物与实际运行代码已冻结到：

`data/experiments/snapshots/pilot_9_pre_evaluator_fix_20260924_150916/`

快照包含：批次汇总、6 个样本的逐轮修复报告、全部初始/修复 San 文件、批次配置，以及 SSM、验证器、轻量渲染器、修复提示和服务端提示构造代码。完整文件哈希见 `SHA256SUMS.txt`。

- Git HEAD：`a943106cd071bd9cae96e8cbf33715fc6a5824b0`
- 配置 SHA-256：`15d5cae17a802f0bd8b88a96423236d7eb0f967124547c773aee725a22b92115`
- 原始汇总 SHA-256：`8ca6e8c9498d99062361e7f6f9edb16196a8529de2f6ecfd1be8428b2bfb93c0`
- 注意：运行时工作区含未提交的 SSM 与批量 Runner 改动，因此快照额外保存了关键代码，不能只依赖 Git commit 复现。

In [ ]:
from pathlib import Path
import hashlib
import json

project_root = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / 'migration_pipeline').is_dir() and (path / 'data').is_dir()
)
snapshot = project_root / 'data/experiments/snapshots/pilot_9_pre_evaluator_fix_20260924_150916'
summary = json.loads((snapshot / 'batch_reports/pilot_9_summary.json').read_text(encoding='utf-8'))
summary['aggregate']

## 2. 修复前原始实验结果

| 复杂度 | 样本 | 首次通过 | 修复轮数 | 最终标记 | 总 token |
|---|---|---:|---:|---:|---:|
| simple | step | 否 | 1 | 通过 | 14,637 |
| simple | BookmarkToggleCard | 否 | 1 | 通过 | 18,570 |
| simple | StarRatingInput | 否 | 1 | 通过 | 22,865 |
| medium | KanbanColumnBoard | 否 | 1 | 通过 | 36,750 |
| medium | DataPaginationTable | 否 | 3 | 未通过 | 94,911 |
| medium | FormRulePlayground | 否 | 3 | 未通过 | 74,660 |

截至暂停时：完成 6/9，最终标记通过 4/6，总 token 为 262,393。simple 标记通过率 100%，medium 标记通过率 33.3%，但由于评估器缺陷，medium 指标不能直接解释为模型真实迁移成功率。

In [ ]:
rows = []
for item in summary['samples']:
    rows.append({
        'id': item['id'],
        'complexity': item['complexity'],
        'final_passed': item.get('final_passed'),
        'repair_rounds': item.get('repair_rounds'),
        'initial_validation_errors': item.get('initial_validation_error_codes', []),
        'initial_visual_errors': item.get('initial_visual_error_codes', []),
        'final_validation_errors': item.get('final_validation_error_codes', []),
        'final_visual_errors': item.get('final_visual_error_codes', []),
        'total_tokens': item.get('total_tokens'),
    })
rows

## 3. 为什么判定存在评估器误判

把数据集中的人工确认参考 San 作为正对照输入同一评估器：

| 正对照 | 静态校验 | 视觉结构校验 | 主要报告 |
|---|---:|---:|---|
| DataPaginationTable 参考 San | 失败 | 失败 | `missing_event_bindings`、computed 执行错误、结构相似度 0.1286 |
| FormRulePlayground 参考 San | 失败 | 失败 | `missing_event_bindings`、模板 methods 不可见、结构相似度 0.6111 |

正对照不应失败。因此，当前失败集合同时包含真实迁移缺陷和测量工具缺陷，不能直接用于比较不同复杂度的迁移能力。

## 4. 已定位的评估器缺陷

1. **事件完整性正则误判带引号参数的 handler**：`setSort('id')`、`updateField('name', $event)` 明明存在于 `on-*`，但旧正则禁止属性表达式内部出现引号。
2. **Vue 轻量渲染器没有把 methods 加入模板表达式作用域**：`isEnabled(rule.id, enabled)` 被报告为未定义。
3. **San 轻量渲染器没有把已计算 computed 值写回 data store**：后续 computed 通过 `this.data.get('sortedRows')` 读取依赖时得到 `undefined`。
4. **Vue/San 条件节点处理不一致**：Vue `v-if=false` 删除节点，而 San `s-if=false` 仅增加隐藏标记，导致节点数量和树编辑距离不可比较。
5. **San 模板方法表达式支持不足**：参考模板中的 `isEnabled(...)` 无法稳定参与属性求值。

## 5. 与误判并存的真实迁移问题

评估器存在问题不代表模型输出正确。人工检查确认：

- **DataPaginationTable**：生成代码使用 `this.computed.sortedRows()` 和 `this.computed.pageCount()`；San 中应通过 `this.data.get('sortedRows')`、`this.data.get('pageCount')` 读取 computed，属于真实运行时缺陷。
- **FormRulePlayground**：最终代码把 `form.age` 从 `18` 改成空字符串，并把 `enabled` 从 `['email', 'adult', 'name']` 改成 `['email', 'adult']`，初始语义发生漂移；错误提示文本也与源组件不一致。
- **跨样本问题**：6 个已运行样本首次生成均出现 `missing_prop_defaults`，说明生成阶段对明确提示的遵循仍不稳定，而修复阶段在 simple/Kanban 上能够恢复。

## 6. 修复与重新实验协议

1. 先增加参考 San 正对照回归测试，再修复验证器和两个轻量渲染器。
2. 验收条件：DataPaginationTable、FormRulePlayground、KanbanColumnBoard 三个参考 San 的静态与视觉评估均通过，且现有回归测试全部通过。
3. 不覆盖本快照；修复后的实验使用新批次 ID `pilot_9_v2` 和独立输出目录。
4. v2 使用相同 9 个样本、相同模型、相同最大修复轮数。若模型或 prompt 改变，必须在 manifest 中单独声明。
5. v2 先 dry-run，再运行 simple/medium；确认评估器稳定后才运行 complex。
6. 论文中将本轮标记为“测量工具校准实验”，不得与 v2 的正式成功率直接合并。

## 7. 评估器修复结果（同日）

已修改：

- `migration_pipeline/stages/validate.py`：解析 `on-*` 属性后提取 handler，不再因字符串参数中的引号误判。
- `migration_pipeline/utils/vue_render.py`：把 Vue methods 暴露给模板表达式。
- `migration_pipeline/utils/san_render.py`：computed 按顺序写回 data store、模板可调用组件方法，并统一 `s-if=false` 的节点移除语义。
- `tests/test_migration_pipeline_regressions.py`：把三个 medium 人工参考 San 固化为正对照。

修复后正对照结果：KanbanColumnBoard、DataPaginationTable、FormRulePlayground 的静态校验、结构相似度、标签序列相似度和文本相似度全部通过，三项相似度均为 1.0；完整回归共 13 项通过。

用修复后的评估器重新评估旧模型产物（未调用模型、未改变旧汇总）：

| 旧模型最终产物 | 静态校验 | 视觉校验 | 结论 |
|---|---:|---:|---|
| KanbanColumnBoard | 通过 | 通过（1.0） | 原通过结论保持 |
| DataPaginationTable | 失败：`vue_this_state` | 失败（0.1714） | `this.computed.*` 等真实缺陷仍被识别 |
| FormRulePlayground | 通过（有提示） | 失败（0.9286） | 初始状态漂移仍造成结构差异 |

这说明修复消除了已知假阳性，同时没有把已确认的模型缺陷放行。

## 8. Pilot 9 v2：Simple/Medium 阶段结论与 Prompt 改进基线

v2 在评估器校准后重新运行了全部 3 个 simple 和 3 个 medium，complex 暂停。结果已冻结到：

`data/experiments/snapshots/pilot_9_v2_pre_prompt_fix_20260924_155149/`

| 复杂度 | 样本 | 首次通过 | 修复轮数 | 最终结果 | Token |
|---|---|---:|---:|---:|---:|
| simple | step | 否 | 1 | 通过 | 14,637 |
| simple | BookmarkToggleCard | 否 | 1 | 通过 | 18,570 |
| simple | StarRatingInput | 是 | 0 | 通过 | 10,568 |
| medium | KanbanColumnBoard | 否 | 1 | 通过 | 36,660 |
| medium | DataPaginationTable | 否 | 3 | 失败 | 94,364 |
| medium | FormRulePlayground | 否 | 3 | 失败 | 72,843 |

累计 247,642 token，最终 4/6 通过。DataPaginationTable 三轮始终保留 `this.computed.sortedRows()` / `this.computed.pageCount()`；现有 repair prompt 没有明确规定 San computed 必须通过 `this.data.get('computedName')` 读取。

FormRulePlayground 第一轮后结构相似度升至 0.9286，但保留了四处属性差异：三个 checkbox 使用不可求值的 `checked="{= isEnabled(...) =}"`，且 `form.age` 从 18 漂移为空字符串；同时 `enabled` 丢失 `name`。第 1、2、3 轮修复文件 SHA-256 完全一致，证明后两轮没有任何代码进展，额外消耗 37,788 token。

下一轮改进应同时处理：computed 访问规则、内部 data 默认值精确保真、派生 checkbox 使用单向插值，以及连续评估签名不变时提前终止修复。v2 不继续运行 complex，避免在已知 repair prompt 缺陷下扩大成本。

In [ ]:
# 校验冻结快照完整性
expected = {}
for line in (snapshot / 'SHA256SUMS.txt').read_text(encoding='utf-8').splitlines():
    digest, relative = line.split('  ', 1)
    expected[relative] = digest

mismatches = []
for relative, digest in expected.items():
    actual = hashlib.sha256((snapshot / relative).read_bytes()).hexdigest()
    if actual != digest:
        mismatches.append((relative, digest, actual))
assert not mismatches, mismatches

manifest = json.loads((snapshot / 'manifest.json').read_text(encoding='utf-8'))
input_mismatches = []
for relative, digest in manifest['input_artifacts'].items():
    actual = hashlib.sha256((project_root / relative).read_bytes()).hexdigest()
    if actual != digest:
        input_mismatches.append((relative, digest, actual))
assert not input_mismatches, input_mismatches
print(f'Integrity verified: {len(expected)} pre-evaluator-fix files and {len(manifest["input_artifacts"])} inputs')

v2_snapshot = project_root / 'data/experiments/snapshots/pilot_9_v2_pre_prompt_fix_20260924_155149'
v2_expected = {}
for line in (v2_snapshot / 'SHA256SUMS.txt').read_text(encoding='utf-8').splitlines():
    digest, relative = line.split('  ', 1)
    v2_expected[relative] = digest
v2_mismatches = []
for relative, digest in v2_expected.items():
    actual = hashlib.sha256((v2_snapshot / relative).read_bytes()).hexdigest()
    if actual != digest:
        v2_mismatches.append((relative, digest, actual))
assert not v2_mismatches, v2_mismatches
print(f'Integrity verified: {len(v2_expected)} pre-prompt-fix v2 files')

## 9. Pilot 9 v3：修复策略改进与实验隔离

针对 v2 暴露的问题，已建立独立配置 `migration_pipeline/batch_configs/pilot_9_v3.json`，保持评估器、样本和最大修复轮数不变，仅改变修复策略，以便后续进行可归因的 v2/v3 对比。

Repair Prompt 新增四项硬约束：San computed 必须通过 `this.data.get('computedName')` 读取；禁止 `this.computed.foo()`；Vue `data()` 的嵌套对象、数组、数字和启用集合必须精确保留；`{= ... =}` 仅用于可赋值路径，方法调用或 computed 等派生表达式使用 `{{ ... }}`。

编排器为每轮修复记录 progress signature，包括规范化代码 SHA-256、validation/visual error codes、树编辑距离及 structure/tag/text similarity。连续两轮签名完全一致且仍未通过时，以 `stop_reason=no_repair_progress` 提前结束；两轮记录均保留在 repair history，便于统计节省的轮数与 token。

本地验证结果：15 项 unittest 全部通过，v3 dry-run 解析出 3 simple、3 medium、3 complex，输出与报告均隔离到 `pilot_9_v3` 目录；dry-run 未创建 summary、输出目录或报告目录，也未调用模型。正式模型实验尚未开始。